In [2]:

import numpy as np
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt


import pickle
import os
import pandas as pd
import kaleido
import plotly.graph_objects as go
import plotly.io as pio

# Force the engine setting
#pio.kaleido.scope.default_format = "svg"
from plotly.subplots import make_subplots
from AnalasysFunction import Split_cal,VolToCalIdx,LongLIST,splitTrace_from_arrays,correct_spikes,_edit_spikes_gui,splitTrace,motorSp,plotVolCal,CalInt,CS_detection,remove_Frame_Multi
from spike_detection_Qixixn2 import complex_bursts_detection,refine_single_spikes,spike_height_calculation,detect_complex_spikes,refine_all_spikes,plot_trace_with_spikes_pdf,plot_trace_with_spikes_export,plot_trace_with_spikes_html
# Create dictionary
import pickle
from matplotlib.widgets import Slider, Button

In [5]:
!pip uninstall -y kaleido
!pip install kaleido==0.1.0post1

Found existing installation: kaleido 1.2.0
Uninstalling kaleido-1.2.0:
  Successfully uninstalled kaleido-1.2.0
   ---------------------------------------- 0.0/56.0 MB ? eta -:--:--
    --------------------------------------- 0.8/56.0 MB 5.6 MB/s eta 0:00:10
   - -------------------------------------- 2.1/56.0 MB 9.8 MB/s eta 0:00:06
   -- ------------------------------------- 4.2/56.0 MB 8.1 MB/s eta 0:00:07
   ---- ----------------------------------- 5.8/56.0 MB 7.8 MB/s eta 0:00:07
   ---- ----------------------------------- 6.3/56.0 MB 7.7 MB/s eta 0:00:07
   ----- ---------------------------------- 7.3/56.0 MB 6.6 MB/s eta 0:00:08
   ----- ---------------------------------- 8.4/56.0 MB 6.4 MB/s eta 0:00:08
   ------ --------------------------------- 9.4/56.0 MB 6.2 MB/s eta 0:00:08
   ------- -------------------------------- 10.2/56.0 MB 5.9 MB/s eta 0:00:08
   ------- -------------------------------- 10.7/56.0 MB 5.6 MB/s eta 0:00:09
   -------- ------------------------------- 11

In [3]:
#functions
def plotAllSpike(TraceV,TraceC,CalAx,VolAX,BurstSId,FinalCom,SingleSId,path,n = 0):
    lBurst = LongLIST(BurstSId)
    lComplex = LongLIST(FinalCom)
    lSingal = LongLIST(SingleSId)
    #lChCalIDX = LongLIST(ChCalIDX)
    #lAmpID = LongLIST(AmpIdx)
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    svg_path =os.path.join(path,f'EventTypeTrace{n}.svg')
    html_path =os.path.join(path,f'EventTypeTrace{n}.html')
    
    fig.add_trace(go.Scatter(x=VolAX, y=TraceV, line=dict(color='chocolate', width=0.8),name="Voltage"),secondary_y=False,)
    if lBurst:
        fig.add_trace(go.Scatter(x=[VolAX[time] for time in lBurst],y=[TraceV[time] for time in lBurst], mode='markers',
                                name="Burst event", marker=dict(color='olivedrab', size=10, symbol='x')),secondary_y=False)
    if lSingal:
        fig.add_trace(go.Scatter(x=[VolAX[time] for time in lSingal],y=[TraceV[time] for time in lSingal], mode='markers',
                                name="Single spike", marker=dict(color='blue', size=10, symbol='x')),secondary_y=False)
    if lComplex:
        fig.add_trace(go.Scatter(x=[VolAX[time] for time in lComplex],y=[TraceV[time] for time in lComplex], mode='markers',
                                name="Complex spike", marker=dict(color='maroon', size=10, symbol='x')),secondary_y=False)
    fig.add_trace(go.Scatter(x= CalAx, y=TraceC ,line=dict(color='blue', width=0.8), name="Calcium"),secondary_y=True,)
    # fig.add_trace(go.Scatter(x= [IntCalXax[time] for time in lChCalIDX], y= [ZscoreCal[time] for time in lChCalIDX], mode='markers',name="chosen clcium range for spiking event",
    #             marker=dict(color='silver', size=10, symbol='circle')), secondary_y=True,)
    # fig.add_trace(go.Scatter(x= [IntCalXax[time] for time in lAmpID], y= [ZscoreCal[time] for time in lAmpID], mode='markers',name="Max clcium chosen for amplitude calculation",
    #             marker=dict(color='dimgrey', size=10, symbol='circle')), secondary_y=True,)
    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",  # Transparent plot area
        paper_bgcolor="rgba(0,0,0,0)",  # Transparent paper (around the plot area)
        xaxis=dict(
            showgrid=False,               # Show grid
            gridcolor='lightgray',       # Grid color for the X-axis
            zerolinecolor="rgba(0,0,0,0)",        # Zero line color
            showline=True,               # Show axis line
            linecolor='black',           # Axis line color
            ticks="outside",             # Ticks outside the plot
        ),
        yaxis=dict(
            showgrid=False,
            gridcolor='lightgray',
            zerolinecolor="rgba(0,0,0,0)",
            showline=True,
            linecolor='black',
            ticks="outside",
        ),
        yaxis2=dict(
            showgrid=False,               # For the secondary Y-axis
            gridcolor='lightgray',
            zerolinecolor="rgba(0,0,0,0)",
            showline=True,
            linecolor='black',
            ticks="outside",
            overlaying='y'               # Overlay on the primary Y-axis
        )
    )  
    # fig.show()
    # Save as SVG
    fig.write_image(svg_path, format="svg")
    # Save as HTML
    fig.write_html(html_path)




In [4]:
def select_params_interactive(trace, spike_idx, frame_rate, init_CS=1.0, init_SS=0.2, init_threshold=0.6):
    """
    Opens an interactive window to tune pnorm_CS, pnorm_SS, and CS_Threshold.
    """
    # --- FIX 1: Close ANY existing windows before starting ---
    plt.close('all') 
    
    # Safety Check
    if len(spike_idx) == 0:
        print("WARNING: Input 'spike_idx' is empty! No spikes to refine.")
        
    selected_params = {'pnorm_CS': init_CS, 'pnorm_SS': init_SS, 'threshold': init_threshold}
    
    # Create figure
    fig, ax = plt.subplots(figsize=(15, 9))
    plt.subplots_adjust(bottom=0.35)
    
    t_axis = np.arange(len(trace)) / frame_rate
    ax.plot(t_axis, trace, 'k', lw=0.5, alpha=0.6, label='Trace')
    
    scat_ss = ax.scatter([], [], c='b', s=30, label='Single Spikes', zorder=5)
    scat_cs = ax.scatter([], [], c='r', s=50, marker='x', label='Complex Spikes', zorder=6)
    
    count_text = ax.text(0.02, 0.95, "Initializing...", transform=ax.transAxes, 
                         bbox=dict(facecolor='white', alpha=0.8))
    burst_spans = []

    ax.set_title(f"Input Spikes: {len(spike_idx)} | Adjust Sliders & Press ENTER")
    ax.legend(loc='upper right')
    
    # Sliders
    ax_cs = plt.axes([0.2, 0.20, 0.6, 0.03])
    ax_ss = plt.axes([0.2, 0.15, 0.6, 0.03])
    ax_th = plt.axes([0.2, 0.10, 0.6, 0.03]) 
    
    slider_cs = Slider(ax_cs, 'P-Norm CS', 0.0, 2, valinit=init_CS)
    slider_ss = Slider(ax_ss, 'P-Norm SS', 0.0, 2, valinit=init_SS)
    slider_th = Slider(ax_th, 'CS Threshold', 0.0, 1, valinit=init_threshold)

    def run_detection(p_cs, p_ss, p_th):
        c_bursts, _ = complex_bursts_detection(trace, spike_idx, frame_rate, 
                                             pnorm=p_cs, process_window=300, plotflag=False)
        r_ss, t_noCS = refine_single_spikes(trace, spike_idx, c_bursts, frame_rate, 
                                          process_window=300, pnorm=p_ss, min_spikes=10, plotflag=False)
        heights, snr = spike_height_calculation(r_ss, trace, c_bursts['trace_mf'], t_noCS, frame_rate)
        cs_spikes_raw = detect_complex_spikes(trace, c_bursts, heights, threshold=p_th, plotflag=False)
        c_bursts_final, r_ss_final, all_cs, all_s = refine_all_spikes(c_bursts, cs_spikes_raw, r_ss)
        return c_bursts_final, r_ss_final, all_cs

    def update(val):
        p_cs = slider_cs.val
        p_ss = slider_ss.val
        p_th = slider_th.val
        selected_params.update({'pnorm_CS': p_cs, 'pnorm_SS': p_ss, 'threshold': p_th})
        
        try:
            c_bursts, r_ss, all_cs = run_detection(p_cs, p_ss, p_th)
            count_text.set_text(f"Single: {len(r_ss)} | Complex: {len(all_cs)}")

            if len(r_ss) > 0:
                idx_ss = r_ss.astype(int)
                scat_ss.set_offsets(np.c_[t_axis[idx_ss], trace[idx_ss]])
                scat_ss.set_visible(True)
            else:
                scat_ss.set_visible(False)

            if len(all_cs) > 0:
                idx_cs = all_cs.astype(int)
                scat_cs.set_offsets(np.c_[t_axis[idx_cs], trace[idx_cs]])
                scat_cs.set_visible(True)
            else:
                scat_cs.set_visible(False)
            
            for span in burst_spans: span.remove()
            burst_spans.clear()
            starts = c_bursts.get('starts', [])
            ends = c_bursts.get('ends', [])
            for s, e in zip(starts, ends):
                span = ax.axvspan(s/frame_rate, e/frame_rate, color='yellow', alpha=0.3)
                burst_spans.append(span)
            fig.canvas.draw_idle()
        except Exception as e:
            print(f"Error in update: {e}")

    slider_cs.on_changed(update)
    slider_ss.on_changed(update)
    slider_th.on_changed(update)
    update(None)
    
    def on_key(event):
        if event.key == 'enter':
            plt.close(fig) 
            plt.close('all') # --- FIX 2: Force close everything on Enter ---

    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.show(block=False)

    # --- SAFE WAIT LOOP ---
    for _ in range(2000):  # 30 seconds max
        if not plt.fignum_exists(fig.number):
            break
        plt.pause(0.1)
    else:
        print("WARNING: Interactive timeout – using current values")

    plt.close('all')

    return (
        selected_params['pnorm_CS'],
        selected_params['pnorm_SS'],
        selected_params['threshold']
    )




def cs_det_interactive_pipeline(path, allT, allC, allSp, frame_rate, 
                                default_pn_CS=0.5, default_pn_SS=0.3, default_thresh=0.5, name=''):
    """
    Main function for Interactive CS Detection.
    """
    process_window_CS = 300
    process_window_SS = 300
    
    all_complex_bursts_dicts = []
    all_refined_SS = []
    all_CS_spikes_list = []
    all_spikes_list = []
    all_spike_heights_interpolated = []
    all_SNR_interpolated = []
    
    used_params_list = []

    
    trace_idx = allT
    
    # Robust spike unwrapping
    raw_spikes = allSp
    if isinstance(raw_spikes, (list, np.ndarray)):
        spike_idx = raw_spikes
    else:
        spike_idx = [raw_spikes]

    print(f"\nProcessing {name} - Cell {1}/{len(allT)}")
    print(" > Opening interactive window...")

    # --- INTERACTIVE STEP ---
    p_CS, p_SS, p_Th = select_params_interactive(trace_idx, spike_idx, frame_rate, 
                                                    init_CS=default_pn_CS, 
                                                    init_SS=default_pn_SS,
                                                    init_threshold=default_thresh)
    
    print(f" > Selected: CS={p_CS:.2f}, SS={p_SS:.2f}, Thresh={p_Th:.2f}")
    used_params_list.append({'pnorm_CS': p_CS, 'pnorm_SS': p_SS, 'threshold': p_Th})
    
    # --- FIX 4: Aggressive Cleanup ---
    # Ensure no windows linger before heavy processing starts
    plt.close('all')
    plt.pause(0.2) 

    # --- FINAL PROCESSING ---
    complex_bursts_dict, _ = complex_bursts_detection(trace_idx, spike_idx, frame_rate, 
                                                        pnorm=p_CS, process_window=process_window_CS, plotflag=False)
    
    refined_SS, trace_noCS = refine_single_spikes(trace_idx, spike_idx, complex_bursts_dict, frame_rate, 
                                                    process_window=process_window_SS, pnorm=p_SS, min_spikes=10, plotflag=False)
    
    heights, snr = spike_height_calculation(refined_SS, trace_idx, complex_bursts_dict['trace_mf'], 
                                                    trace_noCS, frame_rate)
    
    CS_spikes = detect_complex_spikes(trace_idx, complex_bursts_dict, heights, 
                                        threshold=p_Th, plotflag=False)
    
    complex_bursts_dict, refined_SS, all_CS_spikes, all_spikes = refine_all_spikes(complex_bursts_dict, CS_spikes, refined_SS)
    
    # Store results
    all_complex_bursts_dicts.append(complex_bursts_dict)
    all_refined_SS.append(refined_SS)
    all_CS_spikes_list.append(all_CS_spikes)
    all_spikes_list.append(all_spikes)
    all_spike_heights_interpolated.append(heights)
    all_SNR_interpolated.append(snr)
    
    # Save HTML
    save_path_html = os.path.join(path, f'cell_{name}_CS_detection.html')
    
    # --- FIX 5: Ensure the HTML plotter doesn't leave windows open ---
    try:
        plot_trace_with_spikes_html(trace_idx, refined_SS, CS_spikes, complex_bursts_dict, frame_rate, cal=allC, save_path=save_path_html)
        plt.close('all') # Just in case the plotter used matplotlib
    except Exception as e:
        print(f"HTML Save failed: {e}")
    
        # ============================
    # SAVE PKL WITH ALL RESULTS
    # ============================

    save_data = {}

    save_data['trace'] = trace_idx
    save_data['calcium'] = allC
    save_data['input_spikes'] = spike_idx

    save_data['complex_bursts_dicts'] = all_complex_bursts_dicts
    save_data['refined_SS'] = all_refined_SS
    save_data['all_CS_spikes'] = all_CS_spikes_list
    save_data['all_spikes'] = all_spikes_list
    save_data['spike_heights_interpolated'] = all_spike_heights_interpolated
    save_data['SNR_interpolated'] = all_SNR_interpolated

    save_data['CS_detection_params'] = {
        'selected_params_per_cell': used_params_list,
        'process_window_CS': process_window_CS,
        'process_window_SS': process_window_SS,
        'frame_rate': frame_rate
    }

    save_path_pkl = os.path.join(path, f'cell_{name}_CS_detection.pkl')

    try:
        with open(save_path_pkl, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"✓ PKL saved: {save_path_pkl}")
    except Exception as e:
        print(f"✗ PKL save failed: {e}")


    # Return Logic (same as before)
    return {'all_spikes': all_spikes_list, 'refined_SS': all_refined_SS}

In [7]:
l = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\cell1'
path = l
TracePathCal = os.path.join(l,'calTrace.csv')
TracePathVol = os.path.join(l,'volTraceDF.csv')
TracePathCalM = os.path.join(l,'calMask.csv')
TracePathVolM = os.path.join(l,'volMask.csv')
parentP = os.path.dirname(l)
MotPath = os.path.join(parentP,'Sync','MotorId.csv')



VolTrace = pd.read_csv(TracePathVol)
VolTrace = np.array(VolTrace)
VolTrace = VolTrace.flatten()
Trace = VolTrace
motor = pd.read_csv(MotPath, header=None).iloc[:, 0].to_numpy()

CalTrace = pd.read_csv(TracePathCal)
CalTrace = np.array(CalTrace)
CalTrace = CalTrace.flatten()
VolMask = pd.read_csv(TracePathVolM)
VolMask = np.array(VolMask)
VolMask = VolMask.flatten()

#Trace = VolMask 
CalMask = pd.read_csv(TracePathCalM)
bad_frames_vol = np.where(VolMask == False)[0]
bad_frames_cal = np.where(CalMask == False)[0]
CalMask = np.array(CalMask)
CalMask = CalMask.flatten()
VolTrace = Trace
TraceC = CalTrace
VolTrace[bad_frames_vol] = np.nan
TraceC[bad_frames_cal] = np.nan
N = min(len(Trace), len(motor), len(VolMask))

#motor   = motor[:N]
N = min(len(Trace), len(motor), len(VolMask))

motor = np.asarray(motor[:N], dtype=float)   # cast to float so NaN is allowed
motor[bad_frames_vol] = np.nan

motor[bad_frames_vol] = np.nan
#motor= motor[VolMask]

motor_active = motor.any()


VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
CalAX = np.linspace(0, (len(TraceC)/30), len(TraceC))

fpath = os.path.join(l,r'SpikeIdxFinal.csv')
if os.path.exists(fpath):
    pathSpike = fpath
    spikeId = pd.read_csv(pathSpike)
    spikeId = np.array(spikeId)
    spikeId = spikeId.flatten()
    spikeId = spikeId.tolist()
else:
    pathSpike = os.path.join(l,r'SpikeIdx.csv')
    print('LOL')
    IspikeId = pd.read_csv(pathSpike)
    IspikeId = np.array(IspikeId)
    IspikeId = IspikeId.flatten()
    IspikeId = IspikeId.tolist()
    VolMask = VolMask.astype(bool)

    # 2. Create a mapping from Old Index -> New Index
    # This creates an array where every index contains the count of "True" frames before it
    new_mapping = np.cumsum(VolMask) - 1 

    # 3. Filter and Map
    # We keep the spike IF the mask was True at that spot...
    # ...AND we convert it to the new index using our mapping.
    spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]


LOL


In [12]:
#load data
path = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\12-01-2025-awake\fov17\2\cell2'
TracePathCal = os.path.join(path,'calTraceDF.csv')
TracePathVol = os.path.join(path,'volTraceDF.csv')
TracePathSPIKE = os.path.join(path,'SpikeIdx.csv')
VolTrace = pd.read_csv(TracePathVol)
VolTrace = np.array(VolTrace)
VolTrace = VolTrace.flatten()
Trace = VolTrace
CalTrace = pd.read_csv(TracePathCal)
CalTrace = np.array(CalTrace)
CalTrace = CalTrace.flatten()
VolAX = np.linspace(0, (len(Trace)/500), len(Trace)) 
TraceC = CalTrace
CalAX = np.linspace(0, (len(TraceC)/29), len(TraceC))
parentP = os.path.dirname(path)
MotPath = os.path.join(parentP,'Sync','MotorId.csv')
motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
IntCalT,IntCalXax = CalInt(TraceC,CalAX)
spikeId = pd.read_csv(TracePathSPIKE)
spikeId = np.array(spikeId)
spikeId = spikeId.flatten()
spikeId = spikeId.tolist()
motor_active = motor.any()
motor = motor[0:np.size(Trace,0)]



In [12]:
#remove frame
N_Trace,N_TraceC,N_spikeId,N_VolAX,N_CalAX,N_motor,mask_Voltage,mask_calcium=remove_Frame_Multi(Trace,TraceC,spikeId,VolAX,CalAX,motor)
TracePathCal = os.path.join(path,'calMask.csv')
TracePathVol = os.path.join(path,'volMask.csv')
df = pd.DataFrame(mask_calcium, columns=['mask'])  # create df with column name
df.to_csv(TracePathCal, index=False)
df = pd.DataFrame(mask_Voltage, columns=['mask'])  # create df with column name
df.to_csv(TracePathVol, index=False)

z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\AnalasysFunction.py:1393: DeprecationWarning:

invalid escape sequence '\s'

z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\AnalasysFunction.py:1575: DeprecationWarning:

invalid escape sequence '\s'

z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\AnalasysFunction.py:1772: DeprecationWarning:

invalid escape sequence '\s'



KeyboardInterrupt: 

In [71]:
CalAX = N_CalAX
VolTrace = N_Trace
motor = N_motor
TraceC =N_TraceC
VolAX = N_VolAX
spikeId = N_spikeId

In [8]:

## motor stuff

def _safe_corrcoef_local(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    n = min(x.size, y.size)
    if n < 3:
        return -np.inf
    x = x[:n]
    y = y[:n]
    ok = np.isfinite(x) & np.isfinite(y)
    if np.sum(ok) < 3:
        return -np.inf
    x = x[ok]
    y = y[ok]
    sx = float(np.nanstd(x))
    sy = float(np.nanstd(y))
    if sx <= 0 or sy <= 0:
        return -np.inf
    return float(np.corrcoef(x, y)[0, 1])


def _cell_idx_from_folder_local(cell_folder):
    base = os.path.basename(str(cell_folder).rstrip('\/'))
    base_l = base.lower()
    if 'cell' not in base_l:
        return None
    tail = base_l.split('cell')[-1]
    digits = ''.join(ch for ch in tail if ch.isdigit())
    if digits == '':
        return None
    try:
        return int(digits)
    except Exception:
        return None


def _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir):
    f_path = os.path.join(suite2p_dir, 'F.npy')
    if not os.path.isfile(f_path):
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    try:
        F = np.asarray(np.load(f_path, mmap_mode='r'))
    except Exception:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    if F.ndim == 1:
        F = F.reshape(1, -1)
    if F.ndim < 2 or F.shape[0] == 0:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)

    best_idx = None
    best_corr = -np.inf
    for ridx in range(int(F.shape[0])):
        corr = _safe_corrcoef_local(raw_trace, np.asarray(F[int(ridx)], dtype=float).ravel())
        if corr > best_corr:
            best_corr = corr
            best_idx = int(ridx)

    if best_idx is not None and np.isfinite(best_corr):
        return (int(best_idx), float(best_corr))

    cell_idx = _cell_idx_from_folder_local(cell_folder)
    if cell_idx is not None and 0 <= int(cell_idx) < int(F.shape[0]):
        return (int(cell_idx), np.nan)
    return (None, np.nan)


def _build_neuropil_corrected_trace_local(cell_folder, raw_trace, neuropil_r=0.7):
    suite2p_dir = os.path.join(os.path.dirname(cell_folder), 'Sync', 'cal', 'suite2p', 'plane0')
    fneu_path = os.path.join(suite2p_dir, 'Fneu.npy')
    if not os.path.isfile(fneu_path):
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    row_idx, row_corr = _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir)
    if row_idx is None:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    Fneu = np.asarray(np.load(fneu_path, mmap_mode='r'))
    if Fneu.ndim == 1:
        neu = Fneu.ravel().astype(float)
    elif 0 <= int(row_idx) < int(Fneu.shape[0]):
        neu = np.asarray(Fneu[int(row_idx)], dtype=float).ravel()
    else:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    raw = np.asarray(raw_trace, dtype=float).ravel()
    n = min(raw.size, neu.size)
    raw = raw[:n]
    neu = neu[:n]
    cal_nb = raw - float(neuropil_r) * neu

    try:
        pd.DataFrame(cal_nb, columns=['trace']).to_csv(os.path.join(cell_folder, 'calTraceNB.csv'), index=False)
    except Exception:
        pass

    return cal_nb, int(row_idx), float(row_corr)


TraceC_nb, _nb_row_idx, _nb_row_corr = _build_neuropil_corrected_trace_local(path, TraceC, neuropil_r=0.7)
print(f'[motor stuff] neuropil corrected trace ready | row={_nb_row_idx} corr={_nb_row_corr}')

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=VolAX, y=VolTrace, line=dict(color='chocolate', width=0.8), name='Voltage'), secondary_y=False)
fig.add_trace(go.Scatter(x=CalAX, y=TraceC_nb, line=dict(color='blue', width=0.8), name='Calcium NB'), secondary_y=True)
fig.show()

if motor_active:
    volMot, volRest, calMot, calRest, spikeMot, spikeRest, Change_points, MotIdx, calMotID, RestIdx, calRestId = motorSp(TraceC_nb, VolTrace, motor, CalAX, VolAX, spikeId)
    if np.size(RestIdx[0], 0) < 9:
        RestIdx = RestIdx[1:]
        Change_points = Change_points[1:]
        Change_points = Change_points.squeeze()
    changePointPath = os.path.join(path, r'changepoint.csv')
    calMot, calRes, spikeM, spikeR, volM, volR = Split_cal(Change_points, VolTrace, TraceC_nb, VolAX, CalAX, motor, spikeId)
    print(f'calR{len(calRes)}')
    print(f'volR{len(RestIdx)}')
    print(f'calm{len(calMot)}')
    print(f'volm{len(MotIdx)}')
    for i, r in enumerate(calMot):
        calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        pd.DataFrame(r, columns=['trace']).to_csv(calTracPpathMotor, index=False)
        volTracPpathMotor = os.path.join(path, f'volTraceMot{i}.csv')
        pd.DataFrame(volM[i], columns=['trace']).to_csv(volTracPpathMotor, index=False)
        SpiTracPpathMotor = os.path.join(path, f'spikeTraceMot{i}.csv')
        pd.DataFrame(spikeM[i], columns=['trace']).to_csv(SpiTracPpathMotor, index=False)
    for j, m in enumerate(calRes):
        calTracPpathRes = os.path.join(path, f'calTraceRest{j}.csv')
        pd.DataFrame(m, columns=['trace']).to_csv(calTracPpathRes, index=False)
        volTracPpathRest = os.path.join(path, f'volTraceRest{j}.csv')
        pd.DataFrame(volR[j], columns=['trace']).to_csv(volTracPpathRest, index=False)
        SpiTracPpathRest = os.path.join(path, f'spikeTraceRest{j}.csv')
        pd.DataFrame(spikeR[j], columns=['trace']).to_csv(SpiTracPpathRest, index=False)
        print(np.size(m))
    df = pd.DataFrame(np.array(Change_points), columns=['trace'])
    df.to_csv(changePointPath, index=False)

<>:24: DeprecationWarning:

invalid escape sequence '\/'

<>:24: DeprecationWarning:

invalid escape sequence '\/'

C:\Users\owner\AppData\Local\Temp\ipykernel_24916\2974585496.py:24: DeprecationWarning:

invalid escape sequence '\/'



[motor stuff] neuropil corrected trace ready | row=34 corr=0.9994683730544629


calR2
volR5
calm2
volm3
901
1081


In [ ]:
## motor stuff

def _safe_corrcoef_local(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    n = min(x.size, y.size)
    if n < 3:
        return -np.inf
    x = x[:n]
    y = y[:n]
    ok = np.isfinite(x) & np.isfinite(y)
    if np.sum(ok) < 3:
        return -np.inf
    x = x[ok]
    y = y[ok]
    sx = float(np.nanstd(x))
    sy = float(np.nanstd(y))
    if sx <= 0 or sy <= 0:
        return -np.inf
    return float(np.corrcoef(x, y)[0, 1])


def _cell_idx_from_folder_local(cell_folder):
    base = os.path.basename(str(cell_folder).rstrip('\/'))
    base_l = base.lower()
    if 'cell' not in base_l:
        return None
    tail = base_l.split('cell')[-1]
    digits = ''.join(ch for ch in tail if ch.isdigit())
    if digits == '':
        return None
    try:
        return int(digits)
    except Exception:
        return None


def _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir):
    f_path = os.path.join(suite2p_dir, 'F.npy')
    if not os.path.isfile(f_path):
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    try:
        F = np.asarray(np.load(f_path, mmap_mode='r'))
    except Exception:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    if F.ndim == 1:
        F = F.reshape(1, -1)
    if F.ndim < 2 or F.shape[0] == 0:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)

    best_idx = None
    best_corr = -np.inf
    for ridx in range(int(F.shape[0])):
        corr = _safe_corrcoef_local(raw_trace, np.asarray(F[int(ridx)], dtype=float).ravel())
        if corr > best_corr:
            best_corr = corr
            best_idx = int(ridx)

    if best_idx is not None and np.isfinite(best_corr):
        return (int(best_idx), float(best_corr))

    cell_idx = _cell_idx_from_folder_local(cell_folder)
    if cell_idx is not None and 0 <= int(cell_idx) < int(F.shape[0]):
        return (int(cell_idx), np.nan)
    return (None, np.nan)


def _build_neuropil_corrected_trace_local(cell_folder, raw_trace, neuropil_r=0.7):
    suite2p_dir = os.path.join(os.path.dirname(cell_folder), 'Sync', 'cal', 'suite2p', 'plane0')
    fneu_path = os.path.join(suite2p_dir, 'Fneu.npy')
    if not os.path.isfile(fneu_path):
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    row_idx, row_corr = _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir)
    if row_idx is None:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    Fneu = np.asarray(np.load(fneu_path, mmap_mode='r'))
    if Fneu.ndim == 1:
        neu = Fneu.ravel().astype(float)
    elif 0 <= int(row_idx) < int(Fneu.shape[0]):
        neu = np.asarray(Fneu[int(row_idx)], dtype=float).ravel()
    else:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    raw = np.asarray(raw_trace, dtype=float).ravel()
    n = min(raw.size, neu.size)
    raw = raw[:n]
    neu = neu[:n]
    cal_nb = raw - float(neuropil_r) * neu

    try:
        pd.DataFrame(cal_nb, columns=['trace']).to_csv(os.path.join(cell_folder, 'calTraceNB.csv'), index=False)
    except Exception:
        pass

    return cal_nb, int(row_idx), float(row_corr)


TraceC_nb, _nb_row_idx, _nb_row_corr = _build_neuropil_corrected_trace_local(path, TraceC, neuropil_r=0.7)
print(f'[motor stuff] neuropil corrected trace ready | row={_nb_row_idx} corr={_nb_row_corr}')

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=VolAX, y=VolTrace, line=dict(color='chocolate', width=0.8), name='Voltage'), secondary_y=False)
fig.add_trace(go.Scatter(x=CalAX, y=TraceC_nb, line=dict(color='blue', width=0.8), name='Calcium NB'), secondary_y=True)
fig.show()

if motor_active:
    volMot, volRest, calMot, calRest, spikeMot, spikeRest, Change_points, MotIdx, calMotID, RestIdx, calRestId = motorSp(TraceC_nb, VolTrace, motor, CalAX, VolAX, spikeId)
    if np.size(RestIdx[0], 0) < 9:
        RestIdx = RestIdx[1:]
        Change_points = Change_points[1:]
        Change_points = Change_points.squeeze()
    changePointPath = os.path.join(path, r'changepoint.csv')
    calMot, calRes, spikeM, spikeR, volM, volR = Split_cal(Change_points, VolTrace, TraceC_nb, VolAX, CalAX, motor, spikeId)
    print(f'calR{len(calRes)}')
    print(f'volR{len(RestIdx)}')
    print(f'calm{len(calMot)}')
    print(f'volm{len(MotIdx)}')
    for i, r in enumerate(calMot):
        calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        pd.DataFrame(r, columns=['trace']).to_csv(calTracPpathMotor, index=False)
        volTracPpathMotor = os.path.join(path, f'volTraceMot{i}.csv')
        pd.DataFrame(volM[i], columns=['trace']).to_csv(volTracPpathMotor, index=False)
        SpiTracPpathMotor = os.path.join(path, f'spikeTraceMot{i}.csv')
        pd.DataFrame(spikeM[i], columns=['trace']).to_csv(SpiTracPpathMotor, index=False)
    for j, m in enumerate(calRes):
        calTracPpathRes = os.path.join(path, f'calTraceRest{j}.csv')
        pd.DataFrame(m, columns=['trace']).to_csv(calTracPpathRes, index=False)
        volTracPpathRest = os.path.join(path, f'volTraceRest{j}.csv')
        pd.DataFrame(volR[j], columns=['trace']).to_csv(volTracPpathRest, index=False)
        SpiTracPpathRest = os.path.join(path, f'spikeTraceRest{j}.csv')
        pd.DataFrame(spikeR[j], columns=['trace']).to_csv(SpiTracPpathRest, index=False)
        print(np.size(m))
    df = pd.DataFrame(np.array(Change_points), columns=['trace'])
    df.to_csv(changePointPath, index=False)



<>:24: DeprecationWarning:

invalid escape sequence '\/'

<>:24: DeprecationWarning:

invalid escape sequence '\/'

C:\Users\owner\AppData\Local\Temp\ipykernel_36720\2733673535.py:24: DeprecationWarning:

invalid escape sequence '\/'



[motor stuff] neuropil corrected trace ready | row=34 corr=0.9994683730544629


C:\Users\owner\AppData\Local\Temp\ipykernel_36720\2733673535.py:24: DeprecationWarning:

invalid escape sequence '\/'



IndexError: index 0 is out of bounds for axis 0 with size 0

In [89]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\Pyr.csv')
values = DB['SNR'].tolist()
r = DB
frame_rate = 500
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
traces =  []
spikes_volpy = []
calcium = []
#pathPyr = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\2\cell2'
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\19-11-2025-awake\fov20\cell0'
#            ,r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\19-11-2025-awake\fov19\cell0',
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\05-11-2025-motor\fov16\cell2',
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\05-11-2025-motor\fov16\cell0',# the super high firing rate in rest
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\28-10-2025-motor\fov10\cell1',
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\fov2\cell2',#only complex
#            r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\fov2\cell0',#ro much complex
#             r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\FOV1\cell1'
#            ]
for i,l in enumerate([pathPyr[20]]):
    print(l)
    currP =l
    
    
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    motor= motor[VolMask]
    traces.append(TraceV)
    
    calcium.append(TraceC)
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/30), len(TraceC))

    fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    if os.path.exists(fpath):
        pathSpike = fpath
        spikeId = pd.read_csv(pathSpike)
        spikeId = np.array(spikeId)
        spikeId = spikeId.flatten()
        spikeId = spikeId.tolist()
    else:
        pathSpike = os.path.join(l,r'SpikeIdx.csv')
        print('LOL')
        IspikeId = pd.read_csv(pathSpike)
        IspikeId = np.array(IspikeId)
        IspikeId = IspikeId.flatten()
        IspikeId = IspikeId.tolist()
        
        # 1. Ensure VolMask is a boolean numpy array
        VolMask = VolMask.astype(bool)

        # 2. Create a mapping from Old Index -> New Index
        # This creates an array where every index contains the count of "True" frames before it
        new_mapping = np.cumsum(VolMask) - 1 

        # 3. Filter and Map
        # We keep the spike IF the mask was True at that spot...
        # ...AND we convert it to the new index using our mapping.
        spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]


    

    
    
    spikes_volpy.append(spikeId)




#traces = np.array(traces)

Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\2\cell2


In [90]:

pnorm_CS = 0.5
process_window_CS = 300
pnorm_SS = 0.2
process_window_SS = 300
complex_spike_threshold = 0.6# in percent of the spike height
# Lists to store results for all cells
all_complex_bursts_dicts = []
all_refined_SS = []
all_CS_spikes_list = []
all_spikes_list = []
all_spike_heights_interpolated = []
all_SNR_interpolated = []
for idx in range(len(traces)):
    
    currCal = calcium[idx]
    print(f"\n{'='*50}")
    print(f"Processing cell {idx}")
    print(f"{'='*50}")
    main_data_folder = currP
    trace_idx = traces[idx]
    spike_idx = spikes_volpy[idx]
    # complex_bursts_dict, segment_bounds = complex_bursts_detection(trace_idx, spike_idx, frame_rate, pnorm=pnorm_CS, process_window=process_window_CS, plotflag=False)
    # refined_SS, trace_noCS = refine_single_spikes(trace_idx, spike_idx, complex_bursts_dict, frame_rate, process_window=process_window_SS, pnorm=pnorm_SS, min_spikes=10, plotflag=False)
    # spike_heights_interpolated, SNR_interpolated = spike_height_calculation(refined_SS, trace_idx, complex_bursts_dict['trace_mf'], trace_noCS, frame_rate)
    # CS_spikes = detect_complex_spikes(trace_idx, complex_bursts_dict, spike_heights_interpolated, threshold=complex_spike_threshold, plotflag=False)
    # complex_bursts_dict, refined_SS, all_CS_spikes, all_spikes = refine_all_spikes(complex_bursts_dict, CS_spikes, refined_SS)
    save_d_2 = cs_det_interactive_pipeline(currP,trace_idx,currCal,spike_idx,500)
    # Store results for this cell
    # all_complex_bursts_dicts.append(complex_bursts_dict)
    # all_refined_SS.append(refined_SS)
    # all_CS_spikes_list.append(all_CS_spikes)
    # all_spikes_list.append(all_spikes)
    # all_spike_heights_interpolated.append(spike_heights_interpolated)
    # all_SNR_interpolated.append(SNR_interpolated)
    # figure_folder = os.path.join(main_data_folder, 'CS_figures')
    # if not os.path.exists(figure_folder):
    #     os.makedirs(figure_folder)
    # Plot and save as PDF
#     save_path_pdf = os.path.join(currP, f'cell_{idx}_CS_detection.pdf')
#     save_path_html = os.path.join(currP, f'cell_{idx}_CS_detection.html')
#     # delete if exists
#     if os.path.exists(save_path_pdf):
#         os.remove(save_path_pdf)
#     plot_trace_with_spikes_export(trace_idx, refined_SS, CS_spikes, complex_bursts_dict, frame_rate,currCal,
#                                 segment_duration=10, rows_per_page=20,id = idx, save_path=save_path_pdf)
#     plot_trace_with_spikes_html(trace_idx, refined_SS, CS_spikes, complex_bursts_dict, frame_rate,cal=currCal,
#                                  save_path=save_path_html)
#     # Create dictionary
#     save_data = {}
#     save_data['traces'] = trace_idx  
#     save_data['calcium'] = currCal             
#     save_data['spikes_volpy'] = spikes_volpy  
#     save_data['pathPyr'] = pathPyr
#     save_data['complex_bursts_dicts'] = all_complex_bursts_dicts
#     save_data['refined_SS'] = all_refined_SS
#     save_data['all_CS_spikes'] = all_CS_spikes_list
#     save_data['all_spikes'] = all_spikes_list
#     save_data['spike_heights_interpolated'] = all_spike_heights_interpolated
#     save_data['SNR_interpolated'] = all_SNR_interpolated
#     save_data['CS_detection_params'] = {
#         'pnorm_CS': pnorm_CS,
#         'pnorm_SS': pnorm_SS,
#         'process_window_CS': process_window_CS,
#         'process_window_SS': process_window_SS,
#         'complex_spike_threshold': complex_spike_threshold
#     }
#     save_path = os.path.join(main_data_folder, 'merged_aligned_data_CS.pkl')
#     with open(save_path, 'wb') as f:
#         pickle.dump(save_data, f)
# print(f"\n{'='*50}")
# print(f"Saved all results to: {save_path}")
# print(f"{'='*50}")


Processing cell 0

Processing  - Cell 1/54643
 > Opening interactive window...


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\_methods.py:184: RuntimeWarning:

invalid value encountered in divide



 > Selected: CS=0.50, SS=0.30, Thresh=0.50


✓ PKL saved: Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\30-09-2025-MOTOR\FOV1\2\cell2\cell__CS_detection.pkl


In [84]:
## spike detection correction 


traces,intSpikes, startIN, rawTr = splitTrace_from_arrays(VolTrace,spikeId,2)
allSpike  = []
fs = 500
for i,t in enumerate(traces):
    #currS = select_good_cells_with_spike_edit(home_path, fs=500, n_chunks=2)
    currS = correct_spikes(t,fs,intSpikes[i])
    if currS is not None:
        currS = np.asarray(currS) + startIN[i]
        allSpike.extend(currS.tolist())
save_path = os.path.join(path, "SpikeIdxFinal.csv")
pd.DataFrame(allSpike, columns=["spike_index"]).to_csv(save_path, index=False)
spikeId = allSpike

z:\Adam-Lab-Shared\Data\Michal_Rubin\code\AnalasysFunction.py:59: MatplotlibDeprecationWarning:

Setting data with a non sequence type is deprecated since 3.7 and will be remove two minor releases later



[✓] Saved 154 spikes.


z:\Adam-Lab-Shared\Data\Michal_Rubin\code\AnalasysFunction.py:59: MatplotlibDeprecationWarning:

Setting data with a non sequence type is deprecated since 3.7 and will be remove two minor releases later



[✓] Saved 306 spikes.
[edit] Updated 306 spikes.
[✓] Finalized with 306 spikes.


z:\Adam-Lab-Shared\Data\Michal_Rubin\code\AnalasysFunction.py:59: MatplotlibDeprecationWarning:

Setting data with a non sequence type is deprecated since 3.7 and will be remove two minor releases later

z:\Adam-Lab-Shared\Data\Michal_Rubin\code\AnalasysFunction.py:59: MatplotlibDeprecationWarning:

Setting data with a non sequence type is deprecated since 3.7 and will be remove two minor releases later



[✓] Saved 306 spikes.
[✓] Finalized with 306 spikes.


In [128]:
l = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\05-11-2025-motor\fov16\cell2'
path = l
TracePathCal = os.path.join(l,'calTraceDF.csv')
TracePathVol = os.path.join(l,'volTraceDF.csv')
TracePathCalM = os.path.join(l,'calMask.csv')
TracePathVolM = os.path.join(l,'volMask.csv')
parentP = os.path.dirname(l)
MotPath = os.path.join(parentP,'Sync','MotorId.csv')
VolTrace = pd.read_csv(TracePathVol)
VolTrace = np.array(VolTrace)
VolTrace = VolTrace.flatten()
Trace = VolTrace
motor = pd.read_csv(MotPath, header=None).iloc[:, 0].to_numpy()
CalTrace = pd.read_csv(TracePathCal)
CalTrace = np.array(CalTrace)
CalTrace = CalTrace.flatten()
VolMask = pd.read_csv(TracePathVolM)
VolMask = np.array(VolMask)
VolMask = VolMask.flatten()
#Trace = VolMask 
CalMask = pd.read_csv(TracePathCalM)
CalMask = np.array(CalMask)
CalMask = CalMask.flatten()
VolTrace = Trace[VolMask]
TraceC = CalTrace[CalMask]
N = min(len(Trace), len(motor), len(VolMask))
motor   = motor[:N]
motor= motor[VolMask]
motor_active = motor.any()
VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
CalAX = np.linspace(0, (len(TraceC)/30), len(TraceC))
fpath = os.path.join(l,r'SpikeIdxFinal.csv')
if os.path.exists(fpath):
    pathSpike = fpath
    spikeId = pd.read_csv(pathSpike)
    spikeId = np.array(spikeId)
    spikeId = spikeId.flatten()
    spikeId = spikeId.tolist()
else:
    pathSpike = os.path.join(l,r'SpikeIdx.csv')
    print('LOL')
    IspikeId = pd.read_csv(pathSpike)
    IspikeId = np.array(IspikeId)
    IspikeId = IspikeId.flatten()
    IspikeId = IspikeId.tolist()
    VolMask = VolMask.astype(bool)

    # 2. Create a mapping from Old Index -> New Index
    # This creates an array where every index contains the count of "True" frames before it
    new_mapping = np.cumsum(VolMask) - 1 

    # 3. Filter and Map
    # We keep the spike IF the mask was True at that spot...
    # ...AND we convert it to the new index using our mapping.
    spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]


In [7]:
## motor stuff

def _safe_corrcoef_local(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    n = min(x.size, y.size)
    if n < 3:
        return -np.inf
    x = x[:n]
    y = y[:n]
    ok = np.isfinite(x) & np.isfinite(y)
    if np.sum(ok) < 3:
        return -np.inf
    x = x[ok]
    y = y[ok]
    sx = float(np.nanstd(x))
    sy = float(np.nanstd(y))
    if sx <= 0 or sy <= 0:
        return -np.inf
    return float(np.corrcoef(x, y)[0, 1])


def _cell_idx_from_folder_local(cell_folder):
    base = os.path.basename(str(cell_folder).rstrip('\/'))
    base_l = base.lower()
    if 'cell' not in base_l:
        return None
    tail = base_l.split('cell')[-1]
    digits = ''.join(ch for ch in tail if ch.isdigit())
    if digits == '':
        return None
    try:
        return int(digits)
    except Exception:
        return None


def _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir):
    f_path = os.path.join(suite2p_dir, 'F.npy')
    if not os.path.isfile(f_path):
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    try:
        F = np.asarray(np.load(f_path, mmap_mode='r'))
    except Exception:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)
    if F.ndim == 1:
        F = F.reshape(1, -1)
    if F.ndim < 2 or F.shape[0] == 0:
        return (_cell_idx_from_folder_local(cell_folder), np.nan)

    best_idx = None
    best_corr = -np.inf
    for ridx in range(int(F.shape[0])):
        corr = _safe_corrcoef_local(raw_trace, np.asarray(F[int(ridx)], dtype=float).ravel())
        if corr > best_corr:
            best_corr = corr
            best_idx = int(ridx)

    if best_idx is not None and np.isfinite(best_corr):
        return (int(best_idx), float(best_corr))

    cell_idx = _cell_idx_from_folder_local(cell_folder)
    if cell_idx is not None and 0 <= int(cell_idx) < int(F.shape[0]):
        return (int(cell_idx), np.nan)
    return (None, np.nan)


def _build_neuropil_corrected_trace_local(cell_folder, raw_trace, neuropil_r=0.7):
    suite2p_dir = os.path.join(os.path.dirname(cell_folder), 'Sync', 'cal', 'suite2p', 'plane0')
    fneu_path = os.path.join(suite2p_dir, 'Fneu.npy')
    if not os.path.isfile(fneu_path):
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    row_idx, row_corr = _resolve_suite2p_row_idx_local(cell_folder, raw_trace, suite2p_dir)
    if row_idx is None:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    Fneu = np.asarray(np.load(fneu_path, mmap_mode='r'))
    if Fneu.ndim == 1:
        neu = Fneu.ravel().astype(float)
    elif 0 <= int(row_idx) < int(Fneu.shape[0]):
        neu = np.asarray(Fneu[int(row_idx)], dtype=float).ravel()
    else:
        return np.asarray(raw_trace, dtype=float).ravel(), None, np.nan

    raw = np.asarray(raw_trace, dtype=float).ravel()
    n = min(raw.size, neu.size)
    raw = raw[:n]
    neu = neu[:n]
    cal_nb = raw - float(neuropil_r) * neu

    try:
        pd.DataFrame(cal_nb, columns=['trace']).to_csv(os.path.join(cell_folder, 'calTraceNB.csv'), index=False)
    except Exception:
        pass

    return cal_nb, int(row_idx), float(row_corr)


TraceC_nb, _nb_row_idx, _nb_row_corr = _build_neuropil_corrected_trace_local(path, TraceC, neuropil_r=0.7)
print(f'[motor stuff] neuropil corrected trace ready | row={_nb_row_idx} corr={_nb_row_corr}')

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=VolAX, y=VolTrace, line=dict(color='chocolate', width=0.8), name='Voltage'), secondary_y=False)
fig.add_trace(go.Scatter(x=CalAX, y=TraceC_nb, line=dict(color='blue', width=0.8), name='Calcium NB'), secondary_y=True)
fig.show()

if motor_active:
    volMot, volRest, calMot, calRest, spikeMot, spikeRest, Change_points, MotIdx, calMotID, RestIdx, calRestId = motorSp(TraceC_nb, VolTrace, motor, CalAX, VolAX, spikeId)
    if np.size(RestIdx[0], 0) < 9:
        RestIdx = RestIdx[1:]
        Change_points = Change_points[1:]
        Change_points = Change_points.squeeze()
    changePointPath = os.path.join(path, r'changepoint.csv')
    calMot, calRes, spikeM, spikeR, volM, volR = Split_cal(Change_points, VolTrace, TraceC_nb, VolAX, CalAX, motor, spikeId)
    print(f'calR{len(calRes)}')
    print(f'volR{len(RestIdx)}')
    print(f'calm{len(calMot)}')
    print(f'volm{len(MotIdx)}')
    for i, r in enumerate(calMot):
        calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        pd.DataFrame(r, columns=['trace']).to_csv(calTracPpathMotor, index=False)
        volTracPpathMotor = os.path.join(path, f'volTraceMot{i}.csv')
        pd.DataFrame(volM[i], columns=['trace']).to_csv(volTracPpathMotor, index=False)
        SpiTracPpathMotor = os.path.join(path, f'spikeTraceMot{i}.csv')
        pd.DataFrame(spikeM[i], columns=['trace']).to_csv(SpiTracPpathMotor, index=False)
    for j, m in enumerate(calRes):
        calTracPpathRes = os.path.join(path, f'calTraceRest{j}.csv')
        pd.DataFrame(m, columns=['trace']).to_csv(calTracPpathRes, index=False)
        volTracPpathRest = os.path.join(path, f'volTraceRest{j}.csv')
        pd.DataFrame(volR[j], columns=['trace']).to_csv(volTracPpathRest, index=False)
        SpiTracPpathRest = os.path.join(path, f'spikeTraceRest{j}.csv')
        pd.DataFrame(spikeR[j], columns=['trace']).to_csv(SpiTracPpathRest, index=False)
        print(np.size(m))
    df = pd.DataFrame(np.array(Change_points), columns=['trace'])
    df.to_csv(changePointPath, index=False)



calR2
volR2
calm2
volm2
961
807


In [185]:
#spiking properties Pyrmidal
fs = 500
if motor_active:
    motFR = []
    restFR = []
    fig, axes_list = plt.subplots(nrows=(len(MotIdx)+len(RestIdx)+1), ncols=1, figsize=(12, 10), sharex=False)
    for i,l in enumerate(MotIdx):

        curVolMot = VolTrace[l]
        print(len(VolTrace[l]))
        curCalMot = TraceC[np.unique(calMotID[i])]
        StartIDX = l[0]
        EndIdx = l[-1]
        currS = [s for s in spikeId if s > StartIDX and s < EndIdx]
        currS = currS - StartIDX
        #print(currS)
        currSMot = correct_spikes(curVolMot,fs,currS)
        #currSRest = correct_spikes(l,fs,spikeRest)
        VolAXmot = np.linspace(0, (len(curVolMot)/500), len(curVolMot)) 
        CalAXmot = np.linspace(0, (len(curCalMot)/30), len(curCalMot))
       
        motFR.append(len(currSMot)/VolAXmot[-1])
        # VolAXrest = np.linspace(0, (len(volRest)/500), len(volRest)) 
        # CalAXrest = np.linspace(0, (len(calRest)/30), len(calRest))
        TracPpathMotor = os.path.join(path,f'SpikeMot{i}.csv')
        TracPpathVol = os.path.join(path,f'volTraceMot{i}.csv')
        # TracPpathRest = os.path.join(path,'volTraceRes.csv')
        calTracPpathMotor = os.path.join(path,f'calTraceMot{i}.csv')
        csInfoMotor = CS_detection(curVolMot,currSMot,axes_list[i + len(MotIdx)])
        # calTracPpathRest = os.path.join(path,'calTraceRes.csv')
        rows = []
        print(np.max(csInfoMotor['single_spikes']))
        # Single spikes
        for idx in csInfoMotor['single_spikes']:
            rows.append({
                'event': 'single_spike',
                'idx': int(idx),
                'time_s': idx / fs,
            })

        # All bursts
        for burst_list in csInfoMotor['bursts']:
            # burst_list is something like [1500, 1505, 1510]
            
            rows.append({
                'event': 'burst',
                # Save the whole list (Pandas will turn this into a string like "[1500, 1505]" in CSV)
                'indices': burst_list,  
                
                # Calculate time for EVERY spike in the burst
                'all_times_s': [x / fs for x in burst_list],
                
                # It is often useful to still have a single "Start" column for sorting
                'start_time': burst_list[0] / fs
            })

        # Complex bursts
        for burst_list in csInfoMotor['complex_bursts']:
            rows.append({
                'event': 'complex_burst',
                'indices': burst_list,                 # Save the full list of indices
                'all_times_s': [x / fs for x in burst_list], # Save full list of times
                'start_time': burst_list[0] / fs       # Useful for sorting
            })

        # Regular bursts
        for burst_list in csInfoMotor['regular_bursts']:
            rows.append({
                'event': 'regular_burst',
                'indices': burst_list,                 # Save the full list of indices
                'all_times_s': [x / fs for x in burst_list], # Save full list of times
                'start_time': burst_list[0] / fs       # Useful for sorting
            })
        #df = pd.DataFrame(curVolMot, csInfoMotor['burst'], csInfoMotor['complex_bursts'],csInfoMotor['regular_bursts'],csInfoMotor['Vm'],csInfoMotor['single_spikes'],columns=['trace','All_Burst','complex_bursts','regular_bursts','vm','ss'])  # create df with column name
        df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
        df.to_csv(TracPpathMotor, index=False)
        dfVol = pd.DataFrame(np.array(curVolMot), columns=['trace'])
        dfVol.to_csv(TracPpathVol, index=False)
        print(len(curVolMot))
        print(df.shape)
        df = pd.DataFrame(curCalMot, columns=['trace'])  # create df with column name
        df.to_csv(calTracPpathMotor, index=False)
        x=plotAllSpike(curVolMot,curCalMot,CalAXmot,VolAXmot,csInfoMotor['regular_bursts'],csInfoMotor['complex_bursts'],csInfoMotor['single_spikes'],path,i)
        # df = pd.DataFrame(calRest, columns=['trace'])  # create df with column name
        # df.to_csv(calTracPpathRest, index=False)
        #csInfo = CS_detection(curVolMot,currSMot,axes_list[i])
    if np.size(RestIdx[0],0) < 9:
        RestIdx = RestIdx[1:]
    for i,l in enumerate(RestIdx):
        curVolRest = VolTrace[l]
        curCalRest = TraceC[calRestId[i]]
        print(len(VolTrace[l]))
        StartIDX = l[0]
        EndIdx = l[-1]
        currS = [i for i in spikeId if i > StartIDX and i < EndIdx]
        currS = currS - StartIDX
        print(max(currS))
        #currSRe = correct_spikes(l,fs,currS)
        currSRest = correct_spikes(curVolRest,fs,currS)
        print(max(currSRest))
        VolAXRest = np.linspace(0, (len(curVolRest)/500), len(curVolRest)) 
        CalAXRest = np.linspace(0, (len(curCalRest)/30), len(curCalRest))
        restFR.append(len(currSRest)/VolAXRest[-1])
        VolAXrest = np.linspace(0, (len(volRest)/500), len(volRest)) 
        CalAXrest = np.linspace(0, (len(calRest)/30), len(calRest))
        csInfo = CS_detection(curVolRest,currSRest,axes_list[i + len(MotIdx)])
        #TracPpathMotor = os.path.join(path,f'volTraceMot{i}.csv')
        TracPpathRest = os.path.join(path,f'SRes{i}.csv')
        calTracPpathRest = os.path.join(path,f'calTraceResr{i}.csv')
        print(np.max(csInfoMotor['single_spikes']))
        rows = []

        # Single spikes
        for idx in csInfo['single_spikes']:
            rows.append({
                'event': 'single_spike',
                'idx': int(idx),
                'time_s': idx / fs,
            })

        # All bursts
        for burst_list in csInfo['bursts']:
            # burst_list is something like [1500, 1505, 1510]
            
            rows.append({
                'event': 'burst',
                # Save the whole list (Pandas will turn this into a string like "[1500, 1505]" in CSV)
                'indices': burst_list,  
                
                # Calculate time for EVERY spike in the burst
                'all_times_s': [x / fs for x in burst_list],
                
                # It is often useful to still have a single "Start" column for sorting
                'start_time': burst_list[0] / fs
            })

        # Complex bursts
        for burst_list in csInfo['complex_bursts']:
            rows.append({
                'event': 'complex_burst',
                'indices': burst_list,                 # Save the full list of indices
                'all_times_s': [x / fs for x in burst_list], # Save full list of times
                'start_time': burst_list[0] / fs       # Useful for sorting
            })

        # Regular bursts
        for burst_list in csInfo['regular_bursts']:
            rows.append({
                'event': 'regular_burst',
                'indices': burst_list,                 # Save the full list of indices
                'all_times_s': [x / fs for x in burst_list], # Save full list of times
                'start_time': burst_list[0] / fs       # Useful for sorting
            })
        TracPpathRest = os.path.join(path,f'SpikeRest{i}.csv')
        TracPpathVolR = os.path.join(path,f'volTraceRest{i}.csv')
        #df = pd.DataFrame(curVolMot, csInfoMotor['burst'], csInfoMotor['complex_bursts'],csInfoMotor['regular_bursts'],csInfoMotor['Vm'],csInfoMotor['single_spikes'],columns=['trace','All_Burst','complex_bursts','regular_bursts','vm','ss'])  # create df with column name
        df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
        df.to_csv(TracPpathRest, index=False)
        # df = pd.DataFrame(curVolRest, columns=['trace'])  # create df with column name
        # print(len(curVolRest))
        # print(df.shape)
        # df .to_csv(TracPpathVolR, index=False)
        dfVol = pd.DataFrame(np.array(curVolRest), columns=['trace'])
        dfVol.to_csv(TracPpathVolR, index=False)
        print(len(curVolRest))
        print(dfVol.shape)
        #calTracPpathRest = os.path.join(path,'calTraceRes.csv')
        # df = pd.DataFrame(volMot, columns=['trace'])  # create df with column name
        # df.to_csv(TracPpathMotor, index=False)
        #df = pd.DataFrame(curVolRest, csInfo['burst'], csInfo['complex_bursts'],csInfo['regular_bursts'],csInfo['Vm'],csInfo['single_spikes'],columns=['trace','All_Burst','complex_bursts','regular_bursts','vm','ss'])  # create df with column name
        
        #df.to_csv(TracPpathRest, index=False)
        # df = pd.DataFrame(calMot, columns=['trace'])  # create df with column name
        # df.to_csv(calTracPpathMotor, index=False)
        df = pd.DataFrame(curCalRest, columns=['trace'])  # create df with column name
        df.to_csv(calTracPpathRest, index=False)
        
        x=plotAllSpike(curVolRest,curCalRest,CalAXrest,VolAXrest,csInfo['regular_bursts'],csInfo['complex_bursts'],csInfo['single_spikes'],path,i+10)
    FrPpathMotor = os.path.join(path,f'FiringRateMotor.csv')
    df = pd.DataFrame(motFR, columns=['fr'])  # create df with column name
    df.to_csv(FrPpathMotor, index=False)
    FrPpathRest = os.path.join(path,f'FiringRateRest.csv')
    df = pd.DataFrame(restFR, columns=['fr'])  # create df with column name
    df.to_csv(FrPpathRest, index=False)
## need to add awake analsys


if not motor_active:
    fig, axes_list = plt.subplots(nrows=1, ncols=1, figsize=(12, 10), sharex=False)
    frList = []
    #segment_duration = 30 # seconds
    #samples_per_segment = segment_duration * fs # 15,000

    # 3. Calculate number of full segments
    # We use integer division // to see how many full 30s chunks fit
    #num_segments = len(VolTrace) // samples_per_segment

    # 4. Split the data
    # This creates a list of arrays, each 15,000 samples long
    segments = []
  

    segment_duration = 30  # seconds
    samples_per_segment = int(segment_duration * fs)

    N = len(VolTrace)

    for start in range(0, N, samples_per_segment):
        end = min(start + samples_per_segment, N)

        segment = VolTrace[start:end]
        segments.append(segment)

        currS = np.array([s for s in spikeId if start <= s < end]) - start
        fr = len(currS) / ((end - start) / fs)
        frList.append(fr)
    
    
    csInfo = CS_detection(VolTrace,spikeId,axes_list)
    SpikePath= os.path.join(path,f'spikeProp.csv')
    rows = []

    # Single spikes
    for idx in csInfo['single_spikes']:
        rows.append({
            'event': 'single_spike',
            'idx': int(idx),
            'time_s': idx / fs,
        })

    # All bursts
    for burst_list in csInfo['bursts']:
        # burst_list is something like [1500, 1505, 1510]
        
        rows.append({
            'event': 'burst',
            # Save the whole list (Pandas will turn this into a string like "[1500, 1505]" in CSV)
            'indices': burst_list,  
            
            # Calculate time for EVERY spike in the burst
            'all_times_s': [x / fs for x in burst_list],
            
            # It is often useful to still have a single "Start" column for sorting
            'start_time': burst_list[0] / fs
        })

    # Complex bursts
    for burst_list in csInfo['complex_bursts']:
        rows.append({
            'event': 'complex_burst',
            'indices': burst_list,                 # Save the full list of indices
            'all_times_s': [x / fs for x in burst_list], # Save full list of times
            'start_time': burst_list[0] / fs       # Useful for sorting
        })

    # Regular bursts
    for burst_list in csInfo['regular_bursts']:
        rows.append({
            'event': 'regular_burst',
            'indices': burst_list,                 # Save the full list of indices
            'all_times_s': [x / fs for x in burst_list], # Save full list of times
            'start_time': burst_list[0] / fs       # Useful for sorting
        })
    x=plotAllSpike(VolTrace,TraceC,CalAX,VolAX,csInfo['regular_bursts'],csInfo['complex_bursts'],csInfo['single_spikes'],path)
    df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
    df.to_csv(SpikePath, index=False)
    # df = pd.DataFrame({
    #     'All_Burst': csInfo['bursts'],
    #     'complex_bursts': csInfo['complex_bursts'],
    #     'regular_bursts': csInfo['regular_bursts'],
    #     'vm': csInfo['Vm'],
    #     'ss': csInfo['single_spikes'],
    # })  # create df with column name
    #df.to_csv(SpikePath, index=False)
    VmPath= os.path.join(path,f'Vm.csv')
    df = pd.DataFrame({
        'vm': csInfo['Vm']
    })
    df.to_csv(VmPath, index=False)
    FrPath= os.path.join(path,f'FiringRate.csv')
    df = pd.DataFrame(frList, columns=['fr'])  # create df with column name
    df.to_csv(FrPath, index=False)


Skipping burst 18
Skipping burst 20
Skipping burst 42
Skipping burst 49


In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Parameters
fs = 500
s = 500  # Just keeping your variable, though not explicitly used in snippet

if motor_active:
    motFR = []
    restFR = []
    
    # Create one big figure with enough rows for Motor + Rest traces
    # Note: Added +1 just in case, or you can adjust based on exact count
    total_plots = len(MotIdx) + len(RestIdx)
    if total_plots == 0: total_plots = 1
    
    fig, axes_list = plt.subplots(nrows=total_plots, ncols=1, figsize=(12, total_plots*2), sharex=False)
    
    # Ensure axes_list is iterable if there's only 1 plot
    if total_plots == 1:
        axes_list = [axes_list]

    # --- 1. MOTOR LOOP ---
    for i, l in enumerate(MotIdx):
        # Slice Data
        curVolMot = VolTrace[l]
        # Handle Calcium slicing (assuming calMotID matches indices)
        curCalMot = TraceC[np.unique(calMotID[i])]
        
        StartIDX = l[0]
        EndIdx = l[-1]
        
        # Select and correct spikes
        currS = [s for s in spikeId if s > StartIDX and s < EndIdx]
        currS = np.array(currS) - StartIDX
        # print(currS) # Debug if needed
        
        currSMot = correct_spikes(curVolMot, fs, currS)
        
        # Time Axes
        VolAXmot = np.linspace(0, (len(curVolMot)/fs), len(curVolMot)) 
        CalAXmot = np.linspace(0, (len(curCalMot)/30), len(curCalMot)) # Assuming 30Hz for Ca
        
        # Firing Rate Calculation
        motFR.append(len(currSMot)/VolAXmot[-1])
        
        # Paths
        TracPpathMotor = os.path.join(path, f'SpikeMot{i}.csv')
        TracPpathVol = os.path.join(path, f'volTraceMot{i}.csv')
        calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        
        # CS Detection
        # Use the specific axis for this motor segment
        csInfoMotor = CS_detection(curVolMot, currSMot, axes_list[i])
        
        # Collect Rows for CSV
        rows = []

        # A. Single Spikes
        for idx in csInfoMotor['single_spikes']:
            rows.append({
                'event': 'single_spike',
                'idx': int(idx),
                'time_s': idx / fs,
            })

        # B. All Bursts
        # for burst_list in csInfoMotor['bursts']:
        #     rows.append({
        #         'event': 'burst',
        #         'idx': int(burst_list[0]),         # <--- ADDED for sorting
        #         'indices': burst_list, 
        #         'all_times_s': [x / fs for x in burst_list],
        #         'start_time': burst_list[0] / fs
        #     })

        # C. Complex Bursts
        for burst_list in csInfoMotor['complex_bursts']:
            rows.append({
                'event': 'complex_burst',
                'idx': int(burst_list[0]),         # <--- ADDED for sorting
                'indices': burst_list,
                'all_times_s': [x / fs for x in burst_list],
                'start_time': burst_list[0] / fs
            })

        # D. Regular Bursts
        for burst_list in csInfoMotor['regular_bursts']:
            rows.append({
                'event': 'burst',
                'idx': int(burst_list[0]),         # <--- ADDED for sorting
                'indices': burst_list,
                'all_times_s': [x / fs for x in burst_list],
                'start_time': burst_list[0] / fs
            })

        # Save Spike Data
        df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
        df.to_csv(TracPpathMotor, index=False)
        
        # Save Traces
        pd.DataFrame(np.array(curVolMot), columns=['trace']).to_csv(TracPpathVol, index=False)
        x = pd.DataFrame(np.array(curVolMot), columns=['trace'])
        print(x.shape)
        print(np.size(curVolMot))
        
        # Plot
        plotAllSpike(curVolMot, curCalMot, CalAXmot, VolAXmot, 
                     csInfoMotor['regular_bursts'], csInfoMotor['complex_bursts'], 
                     csInfoMotor['single_spikes'], path, i)


    # --- 2. REST LOOP ---
    # Optional filtering of rest indices
    if np.size(RestIdx[0], 0) < 9:
        RestIdx = RestIdx[1:]
        
    for i, l in enumerate(RestIdx):
        # Slice Data
        curVolRest = VolTrace[l]
        curCalRest = TraceC[calRestId[i]]
        
        StartIDX = l[0]
        EndIdx = l[-1]
        
        # Select and correct spikes
        currS = [s for s in spikeId if s > StartIDX and s < EndIdx]
        currS = np.array(currS) - StartIDX
        
        currSRest = correct_spikes(curVolRest, fs, currS)
        
        # Time Axes
        VolAXRest = np.linspace(0, (len(curVolRest)/fs), len(curVolRest)) 
        CalAXRest = np.linspace(0, (len(curCalRest)/30), len(curCalRest))
        
        # Firing Rate
        restFR.append(len(currSRest)/VolAXRest[-1])
        
        # Paths
        TracPpathRest = os.path.join(path, f'SpikeRest{i}.csv')
        TracPpathVolR = os.path.join(path, f'volTraceRest{i}.csv')
        
        
        # CS Detection
        # Use axis shifted by len(MotIdx) so we plot below the motor ones
        csInfo = CS_detection(curVolRest, currSRest, axes_list[i + len(MotIdx)])
        
        # Collect Rows
        rows = []

        # A. Single Spikes
        for idx in csInfo['single_spikes']:
            rows.append({
                'event': 'single_spike',
                'idx': int(idx),
                'time_s': idx / fs,
            })

        # B. All Bursts
        for burst_list in csInfo['bursts']:
            rows.append({
                'event': 'burst',
                'idx': int(burst_list[0]),         # <--- ADDED
                'indices': burst_list,  
                'all_times_s': [x / fs for x in burst_list],
                'start_time': burst_list[0] / fs
            })

        # C. Complex Bursts
        for burst_list in csInfo['complex_bursts']:
            rows.append({
                'event': 'complex_burst',
                'idx': int(burst_list[0]),         # <--- ADDED
                'indices': burst_list,
                'all_times_s': [x / fs for x in burst_list],
                'start_time': burst_list[0] / fs
            })

        # D. Regular Bursts
        for burst_list in csInfo['regular_bursts']:
            rows.append({
                'event': 'regular_burst',
                'idx': int(burst_list[0]),         # <--- ADDED
                'indices': burst_list,
                'all_times_s': [x / fs for x in burst_list],
                'start_time': burst_list[0] / fs
            })

        # Save Spike Data
        df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
        df.to_csv(TracPpathRest, index=False)
        
        # Save Traces
        pd.DataFrame(np.array(curVolRest), columns=['trace']).to_csv(TracPpathVolR, index=False)
        
        #pd.DataFrame(curCalRest, columns=['trace']).to_csv(calTracPpathRest, index=False)
        
        # Plot
        plotAllSpike(curVolRest, curCalRest, CalAXRest, VolAXRest, 
                     csInfo['regular_bursts'], csInfo['complex_bursts'], 
                     csInfo['single_spikes'], path, i + 10) # i+10 to avoid overwriting motor files if naming logic uses index

    # --- SAVE SUMMARY FR ---
    pd.DataFrame(motFR, columns=['fr']).to_csv(os.path.join(path, 'FiringRateMotor.csv'), index=False)
    pd.DataFrame(restFR, columns=['fr']).to_csv(os.path.join(path, 'FiringRateRest.csv'), index=False)


# ==========================================
# IF NOT MOTOR ACTIVE (Whole Trace Analysis)
# ==========================================
if not motor_active:
    fig, axes_list = plt.subplots(nrows=1, ncols=1, figsize=(12, 10), sharex=False)
    
    # 1. Calculate Sliding Window Firing Rate
    frList = []
    segments = []
    segment_duration = 30  # seconds
    samples_per_segment = int(segment_duration * fs)
    N = len(VolTrace)

    for start in range(0, N, samples_per_segment):
        end = min(start + samples_per_segment, N)
        segment = VolTrace[start:end]
        segments.append(segment)
        
        # Filter spikes in this window
        currS = np.array([s for s in spikeId if start <= s < end]) - start
        
        duration = (end - start) / fs
        if duration > 0:
            fr = len(currS) / duration
            frList.append(fr)
    
    # 2. CS Detection on FULL Trace
    csInfo = CS_detection(VolTrace, spikeId, axes_list)
    SpikePath = os.path.join(path, 'spikeProp.csv')
    
    rows = []

    # A. Single Spikes
    for idx in csInfo['single_spikes']:
        rows.append({
            'event': 'single_spike',
            'idx': int(idx),
            'time_s': idx / fs,
        })

    # # B. All Bursts
    # for burst_list in csInfo['bursts']:
    #     rows.append({
    #         'event': 'burst',
    #         'idx': int(burst_list[0]),             # <--- ADDED
    #         'indices': burst_list,  
    #         'all_times_s': [x / fs for x in burst_list],
    #         'start_time': burst_list[0] / fs
    #     })

    # C. Complex Bursts
    for burst_list in csInfo['complex_bursts']:
        rows.append({
            'event': 'complex_burst',
            'idx': int(burst_list[0]),             # <--- ADDED
            'indices': burst_list,
            'all_times_s': [x / fs for x in burst_list],
            'start_time': burst_list[0] / fs
        })

    # D. Regular Bursts
    for burst_list in csInfo['regular_bursts']:
        rows.append({
            'event': 'burst',
            'idx': int(burst_list[0]),             # <--- ADDED
            'indices': burst_list,
            'all_times_s': [x / fs for x in burst_list],
            'start_time': burst_list[0] / fs
        })

    # 3. Create Axes for Plotting (Needed for plotAllSpike)
    VolAX = np.linspace(0, len(VolTrace)/fs, len(VolTrace))
    # Assuming TraceC exists globally. If undefined, you might need to handle it.
    if 'TraceC' in locals():
        CalAX = np.linspace(0, len(TraceC)/30, len(TraceC))
    else:
        # Fallback if no Calcium
        TraceC = np.zeros_like(VolTrace) # Dummy
        CalAX = VolAX 

    # 4. Save and Plot
    x = plotAllSpike(VolTrace, TraceC, CalAX, VolAX, 
                     csInfo['regular_bursts'], csInfo['complex_bursts'], 
                     csInfo['single_spikes'], path)
    
    # Save Spikes sorted by index
    df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
    df.to_csv(SpikePath, index=False)

    # Save Vm
    VmPath = os.path.join(path, 'Vm.csv')
    pd.DataFrame({'vm': csInfo['Vm']}).to_csv(VmPath, index=False)
    
    # Save Firing Rate
    FrPath = os.path.join(path, 'FiringRate.csv')
    pd.DataFrame(frList, columns=['fr']).to_csv(FrPath, index=False)

In [73]:
#BETTERsst

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Parameters
fs = 500
s = 500  # Just keeping your variable, though not explicitly used in snippet

if motor_active:
    motFR = []
    restFR = []
    
    # Create one big figure with enough rows for Motor + Rest traces
    # Note: Added +1 just in case, or you can adjust based on exact count
    total_plots = len(MotIdx) + len(RestIdx)
    if total_plots == 0: total_plots = 1
    
    fig, axes_list = plt.subplots(nrows=total_plots, ncols=1, figsize=(12, total_plots*2), sharex=False)
    
    # Ensure axes_list is iterable if there's only 1 plot
    if total_plots == 1:
        axes_list = [axes_list]

    # --- 1. MOTOR LOOP ---
    for i, l in enumerate(MotIdx):
        # Slice Data
        curVolMot = VolTrace[l]
        # Handle Calcium slicing (assuming calMotID matches indices)
        curCalMot = TraceC[np.unique(calMotID[i])]
        
        StartIDX = l[0]
        EndIdx = l[-1]
        
        # Select and correct spikes
        currS = [s for s in spikeId if s > StartIDX and s < EndIdx]
        currS = np.array(currS) - StartIDX
        # print(currS) # Debug if needed
        currSMot = currS
        #currSMot = correct_spikes(curVolMot, fs, currS)
        
        # Time Axes
        VolAXmot = np.linspace(0, (len(curVolMot)/fs), len(curVolMot)) 
        CalAXmot = np.linspace(0, (len(curCalMot)/30), len(curCalMot)) # Assuming 30Hz for Ca
        
        # Firing Rate Calculation
        motFR.append(len(currSMot)/VolAXmot[-1])
        
        # Paths
        TracPpathMotor = os.path.join(path, f'SpikeMot{i}.csv')
        TracPpathVol = os.path.join(path, f'volTraceMot{i}.csv')
        #calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        
        # CS Detection
        # Use the specific axis for this motor segment
        #csInfoMotor = CS_detection(curVolMot, currSMot, axes_list[i])
        
        # Collect Rows for CSV
        # rows = []

        # A. Single Spikes
        

        # Save Spike Data
        # df = pd.DataFrame(rows), columns=['spike'])
        # df.to_csv(TracPpathMotor, index=False)
        
        # Save Traces
        pd.DataFrame(np.array(curVolMot), columns=['trace']).to_csv(TracPpathVol, index=False)
        pd.DataFrame(np.array(currSMot), columns=['idx']).to_csv(TracPpathMotor, index=False)
        
        # Plot
        # plotAllSpike(curVolMot, curCalMot, CalAXmot, VolAXmot, 
        #              csInfoMotor['regular_bursts'], csInfoMotor['complex_bursts'], 
        #              csInfoMotor['single_spikes'], path, i)


    # --- 2. REST LOOP ---
    # Optional filtering of rest indices
    if np.size(RestIdx[0], 0) < 9:
        RestIdx = RestIdx[1:]
        
    for i, l in enumerate(RestIdx):
        # Slice Data
        curVolRest = VolTrace[l]
        curCalRest = TraceC[calRestId[i]]
        
        StartIDX = l[0]
        EndIdx = l[-1]
        
        # Select and correct spikes
        currS = [s for s in spikeId if s > StartIDX and s < EndIdx]
        currS = np.array(currS) - StartIDX
        currSRest =currS
        #currSRest = correct_spikes(curVolRest, fs, currS)
        
        # Time Axes
        VolAXRest = np.linspace(0, (len(curVolRest)/fs), len(curVolRest)) 
        CalAXRest = np.linspace(0, (len(curCalRest)/30), len(curCalRest))
        
        # Firing Rate
        restFR.append(len(currSRest)/VolAXRest[-1])
        
        # Paths
        TracPpathRest = os.path.join(path, f'SpikeRest{i}.csv')
        TracPpathVolR = os.path.join(path, f'volTraceRest{i}.csv')
        
        
        # CS Detection
        # Use axis shifted by len(MotIdx) so we plot below the motor ones
        #csInfo = CS_detection(curVolRest, currSRest, axes_list[i + len(MotIdx)])
        
        # Collect Rows
       
       

        # Save Spike Data
        # df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
        # df.to_csv(TracPpathRest, index=False)
        
        # Save Traces
        pd.DataFrame(np.array(curVolRest), columns=['trace']).to_csv(TracPpathVolR, index=False)
        
        pd.DataFrame(np.array(currSRest), columns=['idx']).to_csv(TracPpathRest, index=False)
        #pd.DataFrame(curCalRest, columns=['trace']).to_csv(calTracPpathRest, index=False)
        
        # # Plot
        # plotAllSpike(curVolRest, curCalRest, CalAXRest, VolAXRest, 
        #              csInfo['regular_bursts'], csInfo['complex_bursts'], 
        #              csInfo['single_spikes'], path, i + 10) # i+10 to avoid overwriting motor files if naming logic uses index

    # --- SAVE SUMMARY FR ---
    pd.DataFrame(motFR, columns=['fr']).to_csv(os.path.join(path, 'FiringRateMotor.csv'), index=False)
    pd.DataFrame(restFR, columns=['fr']).to_csv(os.path.join(path, 'FiringRateRest.csv'), index=False)


# ==========================================
# IF NOT MOTOR ACTIVE (Whole Trace Analysis)
# ==========================================
if not motor_active:
    fig, axes_list = plt.subplots(nrows=1, ncols=1, figsize=(12, 10), sharex=False)
    
    # 1. Calculate Sliding Window Firing Rate
    frList = []
    segments = []
    segment_duration = 30  # seconds
    samples_per_segment = int(segment_duration * fs)
    N = len(VolTrace)

    for start in range(0, N, samples_per_segment):
        end = min(start + samples_per_segment, N)
        segment = VolTrace[start:end]
        segments.append(segment)
        
        # Filter spikes in this window
        currS = np.array([s for s in spikeId if start <= s < end]) - start
        
        duration = (end - start) / fs
        if duration > 0:
            fr = len(currS) / duration
            frList.append(fr)
    
    # 2. CS Detection on FULL Trace
    #csInfo = CS_detection(VolTrace, spikeId, axes_list)
    SpikePath = os.path.join(path, 'spikeProp.csv')
    
    rows = []

    

    # 3. Create Axes for Plotting (Needed for plotAllSpike)
    # VolAX = np.linspace(0, len(VolTrace)/fs, len(VolTrace))
    # # Assuming TraceC exists globally. If undefined, you might need to handle it.
    # if 'TraceC' in locals():
    #     CalAX = np.linspace(0, len(TraceC)/30, len(TraceC))
    # else:
    #     # Fallback if no Calcium
    #     TraceC = np.zeros_like(VolTrace) # Dummy
    #     CalAX = VolAX 

    # # 4. Save and Plot
    # x = plotAllSpike(VolTrace, TraceC, CalAX, VolAX, 
    #                  csInfo['regular_bursts'], csInfo['complex_bursts'], 
    #                  csInfo['single_spikes'], path)
    
    # Save Spikes sorted by index
    # df = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
    # df.to_csv(SpikePath, index=False)

    # # Save Vm
    # VmPath = os.path.join(path, 'Vm.csv')
    # pd.DataFrame({'vm': csInfo['Vm']}).to_csv(VmPath, index=False)
    
    # Save Firing Rate
    FrPath = os.path.join(path, 'FiringRate.csv')
    pd.DataFrame(frList, columns=['fr']).to_csv(FrPath, index=False)


In [318]:
#SST spiking basic
fs = 500
#motor_active=False
if motor_active:
    motFR = []
    restFR = []
    for i,l in enumerate(MotIdx):
        curVolMot = VolTrace[l]
        curCalMot = TraceC[calMotID[i]]
        StartIDX = l[0]
        EndIdx = l[-1]
        currS = [i for i in spikeId if i > StartIDX and i < EndIdx]
        currS = currS - StartIDX
        #currSMot = correct_spikes(curVolMot,fs,currS)
        #currSRest = correct_spikes(l,fs,spikeRest)
        VolAXmot = np.linspace(0, (len(curVolMot)/500), len(curVolMot)) 
        CalAXmot = np.linspace(0, (len(curCalMot)/30), len(curCalMot))
       
        motFR.append(len(currS)/VolAXmot[-1])
        # VolAXrest = np.linspace(0, (len(volRest)/500), len(volRest)) 
        # CalAXrest = np.linspace(0, (len(calRest)/30), len(calRest))
        TracPpathMotor = os.path.join(path,f'volTraceMot{i}.csv')
        # TracPpathRest = os.path.join(path,'volTraceRes.csv')
        calTracPpathMotor = os.path.join(path,f'calTraceMot{i}.csv')
        
        # calTracPpathRest = os.path.join(path,'calTraceRes.csv')
        df = pd.DataFrame(curVolMot, columns=['trace'])  # create df with column name
        df.to_csv(TracPpathMotor, index=False)
        # df = pd.DataFrame(volRest, columns=['trace'])  # create df with column name
        # df.to_csv(TracPpathRest, index=False)
        df = pd.DataFrame(curCalMot, columns=['trace'])  # create df with column name
        df.to_csv(calTracPpathMotor, index=False)
        # df = pd.DataFrame(calRest, columns=['trace'])  # create df with column name
        # df.to_csv(calTracPpathRest, index=False)
    if np.size(RestIdx[0],0) < 9:
        RestIdx = RestIdx[1:]
    for i,l in enumerate(RestIdx):
        curVolRest = VolTrace[l]
        curCalRest = TraceC[calRestId[i]]
        StartIDX = l[0]
        EndIdx = l[-1]
        currS = [i for i in spikeId if i > StartIDX and i < EndIdx]
        currS = currS - StartIDX
        #currSRe = correct_spikes(l,fs,currS)
        #currSRest = correct_spikes(curVolRest,fs,currS)
        VolAXRest = np.linspace(0, (len(curVolRest)/500), len(curVolRest)) 
        CalAXRest = np.linspace(0, (len(curCalRest)/30), len(curCalRest))
        restFR.append(len(currS)/VolAXRest[-1])
        VolAXrest = np.linspace(0, (len(volRest)/500), len(volRest)) 
        CalAXrest = np.linspace(0, (len(calRest)/30), len(calRest))
        #TracPpathMotor = os.path.join(path,f'volTraceMot{i}.csv')
        TracPpathRest = os.path.join(path,f'volTraceRes{i}.csv')
        calTracPpathRest = os.path.join(path,f'calTraceResr{i}.csv')
        
        #calTracPpathRest = os.path.join(path,'calTraceRes.csv')
        # df = pd.DataFrame(volMot, columns=['trace'])  # create df with column name
        # df.to_csv(TracPpathMotor, index=False)
        df = pd.DataFrame(curVolRest, columns=['trace'])  # create df with column name
        df.to_csv(TracPpathRest, index=False)
        # df = pd.DataFrame(calMot, columns=['trace'])  # create df with column name
        # df.to_csv(calTracPpathMotor, index=False)
        df = pd.DataFrame(curCalRest, columns=['trace'])  # create df with column name
        df.to_csv(calTracPpathRest, index=False)
    FrPpathMotor = os.path.join(path,f'FiringRateMotor.csv')
    df = pd.DataFrame(motFR, columns=['fr'])  # create df with column name
    df.to_csv(FrPpathMotor, index=False)
    FrPpathRest = os.path.join(path,f'FiringRateRest.csv')
    df = pd.DataFrame(restFR, columns=['fr'])  # create df with column name
    df.to_csv(FrPpathRest, index=False)
## need to add awake analsys


if not motor_active:
    frList = []
    #segment_duration = 30 # seconds
    #samples_per_segment = segment_duration * fs # 15,000

    # 3. Calculate number of full segments
    # We use integer division // to see how many full 30s chunks fit
    #num_segments = len(VolTrace) // samples_per_segment

    # 4. Split the data
    # This creates a list of arrays, each 15,000 samples long
    segments = []
  

    segment_duration = 30  # seconds
    samples_per_segment = int(segment_duration * fs)

    N = len(VolTrace)

    for start in range(0, N, samples_per_segment):
        end = min(start + samples_per_segment, N)

        segment = VolTrace[start:end]
        segments.append(segment)

        currS = np.array([s for s in spikeId if start <= s < end]) - start
        fr = len(currS) / ((end - start) / fs)
        frList.append(fr)
    
    
    FrPath= os.path.join(path,f'FiringRate.csv')
    df = pd.DataFrame(frList, columns=['fr'])  # create df with column name
    df.to_csv(FrPath, index=False)